# Prepare selected mushroom inputs

Review assigned to Viviana Pajic; review pending. Prepared with AI assistance. See [AI use](../docs/AI_USE.md).

## Keep the agreed rows

The final test stays separate. Use the fixed search and calibration rows. Noise removal helped, so the selected features exclude the two noise columns.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/review_models.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open this notebook inside the project.')
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / 'mushrooms'), str(ROOT / 'citibike'), str(ROOT)]
import pandas as pd
from IPython.display import display
from search import read_partition, FeatureDesign, preprocessing
frozen = json.loads((ROOT / 'mushrooms/frozen_candidates.json').read_text())
config = frozen['precalibration_selection']['candidate_configurations']['v3_extra_trees_0094']['config']
X, y = read_partition('search')
cal_X, cal_y = read_partition('calibration')
display(pd.Series({'search_rows': len(X), 'calibration_rows': len(cal_X),
                   'reserved_final_test_rows': 1000}))
display(pd.DataFrame({'missing_search_inputs': X.isna().sum()}))

search_rows                 2559
calibration_rows             640
reserved_final_test_rows    1000
dtype: int64

,missing_search_inputs
cap-diameter,1023
stem-height,513
stem-width,197
spore-print-color,2440
gill-color,495
habitat,498
season,510
ring-type,297
cap-shape,215
stem-surface,1730


## Fill gaps inside a training fold

Empty inputs do not mean zero. Fit iterative numeric filling and category encoding only on training rows. Guard ratios against division by zero. This notebook has no graphs.

In [2]:
from folds import cv_folds
train, check = cv_folds(X, y, n_splits=5)[0]
design = FeatureDesign(**config['features'])
training_inputs = design.fit_transform(X.iloc[train])
checking_inputs = design.transform(X.iloc[check])
prepare, _, _ = preprocessing(config, X.iloc[train])
training_matrix = prepare.fit_transform(training_inputs, y.iloc[train])
checking_matrix = prepare.transform(checking_inputs)
print('Training matrix:', training_matrix.shape)
print('Checking matrix:', checking_matrix.shape)
print('No preparation was fitted on checking rows or final-test rows.')

Training matrix: (2047, 69)
Checking matrix: (512, 69)
No preparation was fitted on checking rows or final-test rows.


C:\Users\tomis\CloudAI-research\team-review-2026-10-11\.venv-check\Lib\site-packages\sklearn\impute\_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


## Decision

Keep incomplete rows and the imputer warning. Tree inputs need no standard scaling. The next notebook trains the complete pipeline.

In [3]:
display(config['features'])
display(config['preprocessing'])

{'include_noise': False,
 'engineered': True,
 'missing_counts': False,
 'drop_spore': False}

{'numeric_missing': 'iterative',
 'missing_flags': False,
 'categorical_missing': 'constant',
 'scale': None,
 'encoding': 'onehot',
 'min_frequency': 3}